In [11]:
# --- Imports ---
import ast
import os
import wfdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit



In [15]:
# --- Constants ---
DATA_DIR = "dataset/ECG/dataset1/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"
CSV = DATA_DIR + "ptbxl_database.csv"
SCP = DATA_DIR + "scp_statements.csv"

# --- Load main DataFrame ---
df = pd.read_csv(CSV, index_col="ecg_id")
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

# --- Load SCP statements ---
scp = pd.read_csv(SCP, index_col=0)
scp_diagnosis = scp[scp["diagnostic"] == 1][["diagnostic_class"]]

def aggregated_superclass(code_dict):
    """Aggregate SCP diagnostic classes for a given dictionary of SCP codes."""
    classes = []
    for k in code_dict.keys():
        if k in scp_diagnosis.index:
            cls = scp_diagnosis.loc[k, "diagnostic_class"]
            if isinstance(cls, pd.Series):
                cls = cls.values[0]
            classes.append(cls)
    return list(set(classes)) if classes else ["NORM"]

df["superclass"] = df["scp_codes"].apply(aggregated_superclass)

# Keep single-label entries in superclass (common PTB-XL benchmark)
df = df[df["superclass"].map(len) == 1].copy()
df["superclass"] = df["superclass"].str[0]

print("Records (single label):", df.shape[0])
print("Unique patients:", df["patient_id"].nunique())
print(df["superclass"].value_counts())

Records (single label): 16655
Unique patients: 14898
superclass
NORM    9480
MI      2532
STTC    2400
CD      1708
HYP      535
Name: count, dtype: int64


In [16]:
groups = df["patient_id"].values
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

# Further split train into train/val
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=7)
tr_idx, val_idx = next(gss2.split(train_df, groups=train_df["patient_id"].values))
train_df, val_df = train_df.iloc[tr_idx].copy(), train_df.iloc[val_idx].copy()

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), "patients:", d["patient_id"].nunique())

train 9360 patients: 8379
val 3115 patients: 2794
test 4180 patients: 3725
